# Social Media Sentiment Network Analysis

### SkilledScore — Data Visualization Internship
 
**Supervisor:** Dr. Zeeshan Usmani  
**Intern:** Usman Ali  

---

This project analyzes a **synthetic social media network** to examine user interactions, sentiment patterns, influential users, and community structures through network analysis and interactive visualization.

# Data Preparation

The data preparation stage covers synthetic data generation, sentiment analysis, data cleaning, validation, and export of the raw and cleaned datasets.

## 1. Synthetic Social Media Data Generation

This section generates the mock social media dataset required for sentiment and network analysis.

### Objectives

- Generate **5,000 synthetic users**
- Generate **20,000 interactions** consisting of posts, retweets, and replies
- Create **User ID, Post ID, Timestamp, Interaction Type, and Hashtags**
- Generate synthetic post text for sentiment analysis
- Calculate **Sentiment Score (-1 to 1)** using VADER
- Generate interaction relationships suitable for network graph construction
- Validate the final dataset before analysis

### Initialize the Data Generation Environment

Import the required libraries, establish reproducibility, and define the dataset scale specified for the project.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Fixed seed for reproducible synthetic data
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

N_USERS = 5_000
N_INTERACTIONS = 20_000

### Define Users, Interaction Types, and Hashtags

Establish the user population and categorical components required to generate posts, retweets, and replies across the synthetic social media network.

In [2]:
user_ids = [f"U{i:05d}" for i in range(1, N_USERS + 1)]

interaction_types = ["post", "retweet", "reply"]

hashtags = [
    "#Technology", "#AI", "#DataScience", "#Climate",
    "#Health", "#Education", "#Business", "#Sports",
    "#Entertainment", "#Travel", "#Innovation", "#News"
]

print(f"Unique users: {len(user_ids):,}")
print(f"Interaction types: {interaction_types}")
print(f"Available hashtags: {len(hashtags)}")

Unique users: 5,000
Interaction types: ['post', 'retweet', 'reply']
Available hashtags: 12


### Generate Synthetic Post Content

Create a varied pool of positive, neutral, and negative social media text. These texts will later be scored with **VADER**, rather than assigning sentiment scores manually.

In [3]:
positive_texts = [
    "Absolutely love this new update!",
    "Great progress and amazing results.",
    "This is such a positive development.",
    "Really impressed with the new features.",
    "Excellent work by the entire team.",
    "This made my day!",
    "Very happy with the results.",
    "Fantastic news and great progress."
]

neutral_texts = [
    "The latest update was released today.",
    "Here are the details from today's announcement.",
    "The event is scheduled for this afternoon.",
    "A new report was published this morning.",
    "The latest results are now available.",
    "This topic is being discussed today.",
    "The meeting took place as scheduled.",
    "More information will be available soon."
]

negative_texts = [
    "Really disappointed with this update.",
    "This has been a terrible experience.",
    "The situation is becoming very frustrating.",
    "Very unhappy with the latest changes.",
    "This result is extremely disappointing.",
    "The service has been awful today.",
    "This problem keeps getting worse.",
    "I strongly dislike these changes."
]

text_pool = positive_texts + neutral_texts + negative_texts

print(f"Text templates available: {len(text_pool)}")

Text templates available: 24


### Generate Synthetic Social Media Interactions

Generate 20,000 timestamped posts, retweets, and replies. User-to-user relationships are included for retweets and replies to support network construction and information-flow analysis.

In [4]:
post_ids = [f"P{i:05d}" for i in range(1, N_INTERACTIONS + 1)]

start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 6, 30)
date_range_seconds = int((end_date - start_date).total_seconds())

timestamps = [
    start_date + timedelta(seconds=random.randint(0, date_range_seconds))
    for _ in range(N_INTERACTIONS)
]

interaction_type = np.random.choice(
    interaction_types,
    N_INTERACTIONS,
    p=[0.50, 0.30, 0.20]
)

source_users = np.random.choice(user_ids, N_INTERACTIONS)

# Posts have no target; retweets and replies connect two users
target_users = np.full(N_INTERACTIONS, None, dtype=object)

network_mask = interaction_type != "post"

target_users[network_mask] = np.random.choice(
    user_ids,
    network_mask.sum()
)

# Prevent self-interactions in retweets and replies
self_mask = network_mask & (source_users == target_users)

while self_mask.any():
    target_users[self_mask] = np.random.choice(
        user_ids,
        self_mask.sum()
    )
    self_mask = network_mask & (source_users == target_users)


interactions_df = pd.DataFrame({
    "user_id": source_users,
    "target_user_id": target_users,
    "post_id": post_ids,
    "timestamp": timestamps,
    "interaction_type": interaction_type,
    "hashtags": np.random.choice(hashtags, N_INTERACTIONS),
    "post_text": np.random.choice(text_pool, N_INTERACTIONS)
})

interactions_df = (
    interactions_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

interactions_df.head()

,user_id,target_user_id,post_id,timestamp,interaction_type,hashtags,post_text
0,U00016,U04675,P18195,2026-01-01 00:09:59,retweet,#Education,Great progress and amazing results.
1,U03723,U04674,P15426,2026-01-01 00:23:55,retweet,#News,Absolutely love this new update!
2,U04413,None,P08071,2026-01-01 00:25:49,post,#News,Here are the details from today's announcement.
3,U04170,U03675,P06515,2026-01-01 00:47:27,retweet,#AI,The service has been awful today.
4,U03118,None,P09717,2026-01-01 00:51:32,post,#Technology,This made my day!


### Analyze Sentiment with VADER

Apply VADER sentiment analysis to the synthetic post text and generate a compound sentiment score ranging from **-1 (negative) to +1 (positive)**.

In [5]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Compound score provides the required -1 to +1 sentiment measure
interactions_df["sentiment_score"] = interactions_df["post_text"].apply(
    lambda text: analyzer.polarity_scores(text)["compound"]
)

interactions_df[
    ["post_text", "sentiment_score"]
].head(10)

,post_text,sentiment_score
0,Great progress and amazing results.,0.8934
1,Absolutely love this new update!,0.6989
2,Here are the details from today's announcement.,0.0000
3,The service has been awful today.,-0.4588
4,This made my day!,0.0000
5,The meeting took place as scheduled.,0.0000
6,The event is scheduled for this afternoon.,0.0000
7,This result is extremely disappointing.,-0.5413
8,The latest update was released today.,0.0000
9,A new report was published this morning.,0.0000


### Clean Interactions and Filter Low-Activity Users

Remove duplicate interaction records and filter users with very low activity to improve the quality and readability of the subsequent network analysis.

In [6]:
rows_before = len(interactions_df)

# Remove duplicate interaction records
interactions_clean = interactions_df.drop_duplicates(
    subset=["user_id", "target_user_id", "post_id", "timestamp", "interaction_type"]
).copy()

duplicates_removed = rows_before - len(interactions_clean)

# Measure activity by number of interactions initiated by each user
user_activity = interactions_clean["user_id"].value_counts()

# Retain users with at least 2 interactions
active_users = user_activity[user_activity >= 2].index

interactions_clean = interactions_clean[
    interactions_clean["user_id"].isin(active_users)
].reset_index(drop=True)

print(f"Original interactions: {rows_before:,}")
print(f"Duplicates removed: {duplicates_removed:,}")
print(f"Active users retained: {len(active_users):,}")
print(f"Interactions after filtering: {len(interactions_clean):,}")

Original interactions: 20,000
Duplicates removed: 0
Active users retained: 4,525
Interactions after filtering: 19,622


### Validate the Prepared Dataset

Perform a final audit of dataset structure, sentiment scores, interaction categories, identifiers, and user relationships before network construction.

In [7]:
required_columns = [
    "user_id", "target_user_id", "post_id", "timestamp",
    "sentiment_score", "interaction_type", "hashtags", "post_text"
]

print("FINAL DATA PREPARATION AUDIT")
print("-" * 40)

print(f"Original users: {N_USERS:,}")
print(f"Original interactions: {N_INTERACTIONS:,}")
print(f"Clean interactions: {len(interactions_clean):,}")
print(f"Active source users: {interactions_clean['user_id'].nunique():,}")

print(f"\nColumns present: {all(col in interactions_clean.columns for col in required_columns)}")
print(f"Duplicate Post IDs: {interactions_clean['post_id'].duplicated().sum():,}")
print(f"Missing values (excluding target users): {interactions_clean.drop(columns='target_user_id').isna().sum().sum():,}")

print(
    f"Sentiment range: "
    f"{interactions_clean['sentiment_score'].min():.4f} to "
    f"{interactions_clean['sentiment_score'].max():.4f}"
)

print(
    "Interaction types:",
    sorted(interactions_clean["interaction_type"].unique())
)

network_rows = interactions_clean[
    interactions_clean["interaction_type"].isin(["retweet", "reply"])
]

invalid_targets = network_rows["target_user_id"].isna().sum()
self_interactions = (
    network_rows["user_id"] == network_rows["target_user_id"]
).sum()

print(f"Retweet/reply records: {len(network_rows):,}")
print(f"Missing targets in retweets/replies: {invalid_targets:,}")
print(f"Self-interactions: {self_interactions:,}")

assert all(col in interactions_clean.columns for col in required_columns)
assert interactions_clean["post_id"].duplicated().sum() == 0
assert interactions_clean.drop(columns="target_user_id").isna().sum().sum() == 0
assert interactions_clean["sentiment_score"].between(-1, 1).all()
assert set(interactions_clean["interaction_type"].unique()) == {"post", "retweet", "reply"}
assert invalid_targets == 0
assert self_interactions == 0

print("\nDATA PREPARATION AUDIT PASSED")

FINAL DATA PREPARATION AUDIT
----------------------------------------
Original users: 5,000
Original interactions: 20,000
Clean interactions: 19,622
Active source users: 4,525

Columns present: True
Duplicate Post IDs: 0
Missing values (excluding target users): 0
Sentiment range: -0.7003 to 0.8934
Interaction types: ['post', 'reply', 'retweet']
Retweet/reply records: 9,796
Missing targets in retweets/replies: 0
Self-interactions: 0

DATA PREPARATION AUDIT PASSED


### Export Raw and Cleaned Datasets

Save both the original synthetic dataset and the cleaned analytical dataset for reproducibility and project documentation.

In [8]:
from pathlib import Path

# Save final datasets to the Downloads folder
downloads = Path.home() / "Downloads"

interactions_df.to_csv(
    downloads / "social_media_interactions_raw.csv",
    index=False
)

interactions_clean.to_csv(
    downloads / "social_media_interactions_cleaned.csv",
    index=False
)

print("Datasets exported successfully.")
print(downloads / "social_media_interactions_raw.csv")
print(downloads / "social_media_interactions_cleaned.csv")

Datasets exported successfully.
C:\Users\Adnan LapTop House\Downloads\social_media_interactions_raw.csv
C:\Users\Adnan LapTop House\Downloads\social_media_interactions_cleaned.csv


# Data Visualization

## 2. Network Graph Construction

Construct the social network from cleaned retweet and reply interactions, where users are represented as nodes and interactions between users are represented as edges.

In [9]:
import networkx as nx

# Use retweets and replies because they represent direct user-to-user interactions
network_data = interactions_clean[
    interactions_clean["interaction_type"].isin(["retweet", "reply"])
].copy()

G = nx.DiGraph()

for row in network_data.itertuples(index=False):
    G.add_edge(
        row.user_id,
        row.target_user_id,
        sentiment_score=row.sentiment_score,
        interaction_type=row.interaction_type,
        hashtag=row.hashtags,
        post_id=row.post_id
    )

print(f"Network nodes: {G.number_of_nodes():,}")
print(f"Network edges: {G.number_of_edges():,}")
print(f"Source interaction records: {len(network_data):,}")

Network nodes: 4,873
Network edges: 9,794
Source interaction records: 9,796


### Calculate User Influence

Calculate degree centrality to measure each user's influence within the interaction network. These scores will later determine node sizes in the visualization.

In [10]:
# Calculate degree centrality for each user
degree_centrality = nx.degree_centrality(G)

nx.set_node_attributes(
    G,
    degree_centrality,
    "degree_centrality"
)

centrality_df = pd.DataFrame({
    "user_id": list(degree_centrality.keys()),
    "degree_centrality": list(degree_centrality.values())
}).sort_values(
    "degree_centrality",
    ascending=False
).reset_index(drop=True)

print(f"Users scored: {len(centrality_df):,}")
print(f"Minimum centrality: {centrality_df['degree_centrality'].min():.6f}")
print(f"Maximum centrality: {centrality_df['degree_centrality'].max():.6f}")

centrality_df.head(10)

Users scored: 4,873
Minimum centrality: 0.000205
Maximum centrality: 0.002463


,user_id,degree_centrality
0,U03779,0.002463
1,U04994,0.002463
2,U02306,0.002463
3,U04714,0.002463
4,U03269,0.002258
5,U02616,0.002258
6,U02538,0.002258
7,U00238,0.002258
8,U02223,0.002258
9,U04571,0.002258


### Classify Interaction Sentiment

Classify each network interaction as positive, neutral, or negative using its VADER compound sentiment score. These categories will later control edge colors and sentiment filtering in the interactive network.

In [11]:
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

network_data["sentiment_label"] = network_data["sentiment_score"].apply(
    classify_sentiment
)

print("Network interaction sentiment:")
print(network_data["sentiment_label"].value_counts())

print("\nSentiment percentages:")
print(
    network_data["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Network interaction sentiment:
sentiment_label
Neutral     3636
Negative    3280
Positive    2880
Name: count, dtype: int64

Sentiment percentages:
sentiment_label
Neutral     37.12
Negative    33.48
Positive    29.40
Name: proportion, dtype: float64


### Detect Network Communities

Apply the Louvain community detection algorithm to identify clusters of closely connected users within the social network.

In [12]:
import community as community_louvain

# Louvain operates on an undirected representation of the interaction network
G_undirected = G.to_undirected()

partition = community_louvain.best_partition(
    G_undirected,
    random_state=SEED
)

nx.set_node_attributes(G, partition, "community")

community_counts = pd.Series(partition).value_counts().sort_values(
    ascending=False
)

print(f"Communities detected: {len(community_counts):,}")
print(f"Largest community: {community_counts.iloc[0]:,} users")
print(f"Smallest community: {community_counts.iloc[-1]:,} users")

community_counts.head(10)

Communities detected: 41
Largest community: 278 users
Smallest community: 2 users


37    278
1     266
19    229
24    223
4     212
14    179
20    161
31    160
16    155
10    151
Name: count, dtype: int64

### Create Force-Directed Network Layout

Generate node positions using a force-directed spring layout so that strongly connected users appear closer together while maintaining visual separation across the network.

In [13]:
# Generate reproducible force-directed positions
pos = nx.spring_layout(
    G_undirected,
    seed=SEED,
    k=1 / np.sqrt(G_undirected.number_of_nodes()),
    iterations=10
)

print(f"Node positions generated: {len(pos):,}")
print(f"Expected nodes: {G.number_of_nodes():,}")

assert len(pos) == G.number_of_nodes()

print("FORCE-DIRECTED LAYOUT PASSED")

Node positions generated: 4,873
Expected nodes: 4,873
FORCE-DIRECTED LAYOUT PASSED


### Prepare Interactive Network Visualization

Prepare node sizes and sentiment-based edge groups for the interactive Plotly network visualization.

In [14]:
# Scale node sizes according to degree centrality
max_centrality = max(degree_centrality.values())

node_sizes = {
    node: 6 + (degree_centrality[node] / max_centrality) * 18
    for node in G.nodes()
}

# Separate interactions by sentiment
positive_edges = []
negative_edges = []
neutral_edges = []

for row in network_data.itertuples(index=False):
    edge = (row.user_id, row.target_user_id)

    if row.sentiment_label == "Positive":
        positive_edges.append(edge)
    elif row.sentiment_label == "Negative":
        negative_edges.append(edge)
    else:
        neutral_edges.append(edge)

print(f"Positive edges: {len(positive_edges):,}")
print(f"Negative edges: {len(negative_edges):,}")
print(f"Neutral edges: {len(neutral_edges):,}")
print(f"Node sizes prepared: {len(node_sizes):,}")

assert len(node_sizes) == G.number_of_nodes()
assert len(positive_edges) + len(negative_edges) + len(neutral_edges) == len(network_data)

print("NETWORK VISUALIZATION DATA PASSED")

Positive edges: 2,880
Negative edges: 3,280
Neutral edges: 3,636
Node sizes prepared: 4,873
NETWORK VISUALIZATION DATA PASSED


### Build the Interactive Sentiment Network

Create the Plotly network graph with influence-based node sizing, sentiment-colored edges, community information, and interactive hover details.

In [15]:
# Prepare latest post details for user hover information
latest_interaction = (
    interactions_clean
    .sort_values("timestamp")
    .groupby("user_id")
    .tail(1)
    .set_index("user_id")
)

print(f"Users with post details available: {len(latest_interaction):,}")

Users with post details available: 4,525


In [16]:
import plotly.graph_objects as go

def create_edge_trace(edges, color, name):
    x, y = [], []

    for source, target in edges:
        x0, y0 = pos[source]
        x1, y1 = pos[target]

        x.extend([x0, x1, None])
        y.extend([y0, y1, None])

    return go.Scattergl(
        x=x,
        y=y,
        mode="lines",
        line=dict(width=0.5, color=color),
        hoverinfo="skip",
        name=name
    )


positive_trace = create_edge_trace(
    positive_edges, "green", "Positive"
)

negative_trace = create_edge_trace(
    negative_edges, "red", "Negative"
)

neutral_trace = create_edge_trace(
    neutral_edges, "gray", "Neutral"
)


# Prepare node visualization data
node_x = []
node_y = []
node_size = []
node_color = []
node_hover = []

for node in G.nodes():
    x, y = pos[node]

    node_x.append(x)
    node_y.append(y)
    node_size.append(node_sizes[node])
    node_color.append(partition[node])

    # Add latest post details for this user
    if node in latest_interaction.index:
        post = latest_interaction.loc[node]

        post_details = (
            f"<br>Post ID: {post['post_id']}"
            f"<br>Interaction: {post['interaction_type'].title()}"
            f"<br>Hashtag: {post['hashtags']}"
            f"<br>Sentiment: {post['sentiment_score']:.4f}"
        )
    else:
        post_details = (
            "<br>Post ID: N/A"
            "<br>Interaction: N/A"
            "<br>Hashtag: N/A"
            "<br>Sentiment: N/A"
        )

    node_hover.append(
        f"User: {node}<br>"
        f"Degree Centrality: {degree_centrality[node]:.6f}<br>"
        f"Community: {partition[node]}<br>"
        f"Connections: {G.degree(node)}"
        f"{post_details}"
    )


node_trace = go.Scattergl(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    hovertext=node_hover,
    marker=dict(
        size=node_size,
        color=node_color,
        colorscale="Turbo",
        showscale=True,
        colorbar=dict(title="Community"),
        line=dict(width=0.3)
    ),
    name="Users",
    showlegend=False
)


# Separate fixed-color Users legend marker
users_legend_trace = go.Scattergl(
    x=[None],
    y=[None],
    mode="markers",
    marker=dict(
        size=10,
        color="#FFA15A"
    ),
    name="Users",
    hoverinfo="skip",
    showlegend=True
)


fig = go.Figure(
    data=[
        neutral_trace,
        positive_trace,
        negative_trace,
        node_trace,
        users_legend_trace
    ]
)


# Final presentation layout
milky_cream = "#FAF8F2"
burgundy = "#5A1831"

fig.update_layout(
    title=dict(
        text="<b>Interactive Social Media Sentiment Network</b>",
        x=0.025,
        y=0.965,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=25,
            color=burgundy,
            family="Arial"
        )
    ),

    showlegend=True,
    hovermode="closest",

    # Fits comfortably within a 100% zoom browser window
    autosize=True,
    width=None,
    height=700,

    paper_bgcolor=milky_cream,
    plot_bgcolor=milky_cream,

    # Sentiment legend inside top-right
    legend=dict(
        title=dict(
            text="<b>Sentiment</b>",
            font=dict(size=14, color=burgundy)
        ),
        x=0.91,
        y=0.94,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(250,248,242,0.88)",
        borderwidth=0,
        font=dict(size=12)
    ),

    xaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1
    ),

    margin=dict(
        l=15,
        r=15,
        t=70,
        b=10
    )
)


# Community colorbar positioned at far-right
node_trace.marker.colorbar.update(
    title=dict(
        text="<b>Community</b>",
        font=dict(color=burgundy, size=14)
    ),
    x=0.985,
    y=0.47,
    len=0.68,
    thickness=16,
    xanchor="right"
)


# Interactive sentiment filter
fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.02,
            y=0.90,
            xanchor="left",
            yanchor="top",

            buttons=[
                dict(
                    label="All Sentiments",
                    method="update",
                    args=[{
                        "visible": [True, True, True, True, True]
                    }]
                ),

                dict(
                    label="Positive",
                    method="update",
                    args=[{
                        "visible": [False, True, False, True, True]
                    }]
                ),

                dict(
                    label="Negative",
                    method="update",
                    args=[{
                        "visible": [False, False, True, True, True]
                    }]
                ),

                dict(
                    label="Neutral",
                    method="update",
                    args=[{
                        "visible": [True, False, False, True, True]
                    }]
                )
            ]
        )
    ]
)


fig.show(renderer="browser")

### Visualize the Influential User Network

Create a focused subnetwork of the most influential users to reveal network structure, communities, and interaction patterns that are obscured in the full-scale network.

In [17]:
# Select the 30 users with the highest degree centrality
top_influencers = centrality_df.head(30)["user_id"].tolist()

# Include each influencer and their directly connected neighbors
focus_nodes = set(top_influencers)

for user in top_influencers:
    focus_nodes.update(G.predecessors(user))
    focus_nodes.update(G.successors(user))

# Build the influencer-centered subnetwork
G_focus = G.subgraph(focus_nodes).copy()

# Preserve the exact spatial positions from the full network
pos_focus = {
    node: pos[node]
    for node in G_focus.nodes()
}

assert len(pos_focus) == G_focus.number_of_nodes()

print(f"Core influencers: {len(top_influencers):,}")
print(f"Focused network nodes: {G_focus.number_of_nodes():,}")
print(f"Focused network edges: {G_focus.number_of_edges():,}")
print(f"Focused positions: {len(pos_focus):,}")

Core influencers: 30
Focused network nodes: 333
Focused network edges: 354
Focused positions: 333


### Visualize the Focused Influential User Network

Visualize the focused subnetwork of high-influence users to reveal clearer graph structure, direct connections, community membership, and relative influence that may be obscured in the full-scale network.

In [18]:
import plotly.graph_objects as go


# Prepare focused interaction data
focus_data = network_data[
    network_data["user_id"].isin(G_focus.nodes()) &
    network_data["target_user_id"].isin(G_focus.nodes())
].copy()


# Separate interactions by sentiment
focus_positive_edges = list(
    zip(
        focus_data.loc[
            focus_data["sentiment_label"] == "Positive", "user_id"
        ],
        focus_data.loc[
            focus_data["sentiment_label"] == "Positive", "target_user_id"
        ]
    )
)

focus_negative_edges = list(
    zip(
        focus_data.loc[
            focus_data["sentiment_label"] == "Negative", "user_id"
        ],
        focus_data.loc[
            focus_data["sentiment_label"] == "Negative", "target_user_id"
        ]
    )
)

focus_neutral_edges = list(
    zip(
        focus_data.loc[
            focus_data["sentiment_label"] == "Neutral", "user_id"
        ],
        focus_data.loc[
            focus_data["sentiment_label"] == "Neutral", "target_user_id"
        ]
    )
)


# Create edge traces
def create_focus_edge_trace(edges, color, name):
    x, y = [], []

    for source, target in edges:
        x0, y0 = pos_focus[source]
        x1, y1 = pos_focus[target]

        x.extend([x0, x1, None])
        y.extend([y0, y1, None])

    return go.Scattergl(
        x=x,
        y=y,
        mode="lines",
        line=dict(width=0.5, color=color),
        hoverinfo="skip",
        name=name
    )


focus_positive_trace = create_focus_edge_trace(
    focus_positive_edges, "green", "Positive"
)

focus_negative_trace = create_focus_edge_trace(
    focus_negative_edges, "red", "Negative"
)

focus_neutral_trace = create_focus_edge_trace(
    focus_neutral_edges, "gray", "Neutral"
)


# Prepare node data
focus_node_x = []
focus_node_y = []
focus_node_size = []
focus_node_color = []
focus_node_hover = []

for node in G_focus.nodes():
    x, y = pos_focus[node]

    focus_node_x.append(x)
    focus_node_y.append(y)
    focus_node_size.append(node_sizes[node])
    focus_node_color.append(partition[node])

    # Add latest post details to hover
    if node in latest_interaction.index:
        post = latest_interaction.loc[node]

        post_details = (
            f"<br>Post ID: {post['post_id']}"
            f"<br>Interaction: {post['interaction_type'].title()}"
            f"<br>Hashtag: {post['hashtags']}"
            f"<br>Sentiment: {post['sentiment_score']:.4f}"
        )
    else:
        post_details = (
            "<br>Post ID: N/A"
            "<br>Interaction: N/A"
            "<br>Hashtag: N/A"
            "<br>Sentiment: N/A"
        )

    focus_node_hover.append(
        f"User: {node}<br>"
        f"Degree Centrality: {degree_centrality[node]:.6f}<br>"
        f"Community: {partition[node]}<br>"
        f"Connections: {G.degree(node)}"
        f"{post_details}"
    )


# Create node trace
focus_node_trace = go.Scattergl(
    x=focus_node_x,
    y=focus_node_y,
    mode="markers",
    hoverinfo="text",
    hovertext=focus_node_hover,

    marker=dict(
        size=focus_node_size,
        color=focus_node_color,
        colorscale="Turbo",
        showscale=True,
        colorbar=dict(title="Community"),
        line=dict(width=0.3)
    ),

    name="Users",
    showlegend=False
)


# Fixed Users legend marker
focus_users_legend_trace = go.Scattergl(
    x=[None],
    y=[None],
    mode="markers",

    marker=dict(
        size=10,
        color="#FFA15A"
    ),

    name="Users",
    hoverinfo="skip",
    showlegend=True
)


# Build figure
fig_focus = go.Figure(
    data=[
        focus_neutral_trace,
        focus_positive_trace,
        focus_negative_trace,
        focus_node_trace,
        focus_users_legend_trace
    ]
)


# Presentation layout
milky_cream = "#FAF8F2"
burgundy = "#5A1831"

fig_focus.update_layout(
    title=dict(
        text="<b>Influential User Network</b>",
        x=0.025,
        y=0.965,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=25,
            color=burgundy,
            family="Arial"
        )
    ),

    showlegend=True,
    hovermode="closest",
    autosize=True,
    width=None,
    height=700,

    paper_bgcolor=milky_cream,
    plot_bgcolor=milky_cream,

    legend=dict(
        title=dict(
            text="<b>Sentiment</b>",
            font=dict(
                size=14,
                color=burgundy
            )
        ),
        x=0.91,
        y=0.94,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(250,248,242,0.88)",
        borderwidth=0,
        font=dict(size=12)
    ),

    xaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        visible=False,
        showgrid=False,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1
    ),

    margin=dict(
        l=15,
        r=15,
        t=70,
        b=10
    )
)


# Community colorbar
focus_node_trace.marker.colorbar.update(
    title=dict(
        text="<b>Community</b>",
        font=dict(
            color=burgundy,
            size=14
        )
    ),
    x=0.985,
    y=0.47,
    len=0.68,
    thickness=16,
    xanchor="right"
)


# Sentiment filter
fig_focus.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.02,
            y=0.90,
            xanchor="left",
            yanchor="top",

            buttons=[
                dict(
                    label="All Sentiments",
                    method="update",
                    args=[{
                        "visible": [True, True, True, True, True]
                    }]
                ),

                dict(
                    label="Positive",
                    method="update",
                    args=[{
                        "visible": [False, True, False, True, True]
                    }]
                ),

                dict(
                    label="Negative",
                    method="update",
                    args=[{
                        "visible": [False, False, True, True, True]
                    }]
                ),

                dict(
                    label="Neutral",
                    method="update",
                    args=[{
                        "visible": [True, False, False, True, True]
                    }]
                )
            ]
        )
    ]
)


fig_focus.show(renderer="browser")

### Validate Network Visualization

Validate the full interactive sentiment network and the focused influencer network to ensure that the required network structure, influence metrics, sentiment layers, communities, force-directed layouts, and interactive components were successfully implemented.

In [19]:
print("FINAL VISUALIZATION DESIGN AUDIT")
print("-" * 45)

# ============================================================
# Full Network
# ============================================================

print(f"Full network nodes: {G.number_of_nodes():,}")
print(f"Full network edges: {G.number_of_edges():,}")
print(f"Degree centrality scores: {len(degree_centrality):,}")
print(f"Louvain communities: {len(set(partition.values())):,}")
print(f"Force-directed positions: {len(pos):,}")


# ============================================================
# Sentiment Layers
# ============================================================

print(f"Positive interactions: {len(positive_edges):,}")
print(f"Negative interactions: {len(negative_edges):,}")
print(f"Neutral interactions: {len(neutral_edges):,}")


# ============================================================
# Focused Influencer Network
# ============================================================

print(f"Core influencers: {len(top_influencers):,}")
print(f"Focused network nodes: {G_focus.number_of_nodes():,}")
print(f"Focused network edges: {G_focus.number_of_edges():,}")
print(f"Focused positions: {len(pos_focus):,}")

print(f"Focused positive interactions: {len(focus_positive_edges):,}")
print(f"Focused negative interactions: {len(focus_negative_edges):,}")
print(f"Focused neutral interactions: {len(focus_neutral_edges):,}")


# ============================================================
# Core Network Validation
# ============================================================

assert len(degree_centrality) == G.number_of_nodes()
assert len(partition) == G.number_of_nodes()
assert len(pos) == G.number_of_nodes()
assert len(node_sizes) == G.number_of_nodes()


# ============================================================
# Full Sentiment Validation
# ============================================================

assert (
    len(positive_edges)
    + len(negative_edges)
    + len(neutral_edges)
    == len(network_data)
)


# ============================================================
# Full Interactive Figure Validation
# ============================================================

# 3 sentiment traces + real nodes + Users legend trace
assert len(fig.data) == 5

# Sentiment dropdown exists
assert len(fig.layout.updatemenus) == 1


# ============================================================
# Focused Network Validation
# ============================================================

assert len(top_influencers) == 30
assert len(pos_focus) == G_focus.number_of_nodes()
assert len(focus_node_size) == G_focus.number_of_nodes()

assert G_focus.number_of_nodes() > 0
assert G_focus.number_of_edges() > 0


# All focused interactions belong to one sentiment layer
assert (
    len(focus_positive_edges)
    + len(focus_negative_edges)
    + len(focus_neutral_edges)
    == len(focus_data)
)


# ============================================================
# Focused Interactive Figure Validation
# ============================================================

# 3 sentiment traces + real nodes + Users legend trace
assert len(fig_focus.data) == 5

# Sentiment dropdown exists
assert len(fig_focus.layout.updatemenus) == 1


# ============================================================
# Final Result
# ============================================================

print("\nFINAL VISUALIZATION DESIGN AUDIT PASSED")

FINAL VISUALIZATION DESIGN AUDIT
---------------------------------------------
Full network nodes: 4,873
Full network edges: 9,794
Degree centrality scores: 4,873
Louvain communities: 41
Force-directed positions: 4,873
Positive interactions: 2,880
Negative interactions: 3,280
Neutral interactions: 3,636
Core influencers: 30
Focused network nodes: 333
Focused network edges: 354
Focused positions: 333
Focused positive interactions: 92
Focused negative interactions: 125
Focused neutral interactions: 137

FINAL VISUALIZATION DESIGN AUDIT PASSED


# 3. Analysis & Documentation

## Key Influencers and Sentiment

Identify the most influential users in the network and examine whether their interactions primarily drive positive, negative, or neutral sentiment.

In [20]:
# Summarize sentiment activity for each user
user_sentiment = (
    network_data
    .groupby(["user_id", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for sentiment in ["Positive", "Negative", "Neutral"]:
    if sentiment not in user_sentiment.columns:
        user_sentiment[sentiment] = 0

# Add degree centrality and retain the top 30 influencers
user_sentiment["degree_centrality"] = (
    user_sentiment["user_id"].map(degree_centrality)
)

influencer_sentiment = (
    user_sentiment[
        user_sentiment["user_id"].isin(top_influencers)
    ]
    .sort_values("degree_centrality", ascending=False)
    .reset_index(drop=True)
)

print("KEY INFLUENCER SENTIMENT ANALYSIS")
print("-" * 45)
print(influencer_sentiment.to_string(index=False))

KEY INFLUENCER SENTIMENT ANALYSIS
---------------------------------------------
user_id  Negative  Neutral  Positive  degree_centrality
 U04994         3        5         0           0.002463
 U04714         1        1         3           0.002463
 U03779         3        2         2           0.002463
 U02306         1        2         3           0.002463
 U02538         2        1         2           0.002258
 U04571         4        2         1           0.002258
 U03487         2        1         2           0.002258
 U03269         3        1         2           0.002258
 U03167         0        1         5           0.002258
 U02616         1        1         3           0.002258
 U00238         1        2         1           0.002258
 U02223         3        3         0           0.002258
 U01969         4        4         1           0.002258
 U01269         1        4         2           0.002258
 U02424         3        0         2           0.002053
 U04198         0       

In [21]:
# Identify strongest sentiment-driving influencers
top_positive_influencers = influencer_sentiment.nlargest(
    5, "Positive"
)[["user_id", "Positive", "degree_centrality"]]

top_negative_influencers = influencer_sentiment.nlargest(
    5, "Negative"
)[["user_id", "Negative", "degree_centrality"]]

print("TOP POSITIVE SENTIMENT INFLUENCERS")
print("-" * 40)
print(top_positive_influencers.to_string(index=False))

print("\nTOP NEGATIVE SENTIMENT INFLUENCERS")
print("-" * 40)
print(top_negative_influencers.to_string(index=False))

TOP POSITIVE SENTIMENT INFLUENCERS
----------------------------------------
user_id  Positive  degree_centrality
 U03167         5           0.002258
 U04714         3           0.002463
 U02306         3           0.002463
 U02616         3           0.002258
 U04215         3           0.002053

TOP NEGATIVE SENTIMENT INFLUENCERS
----------------------------------------
user_id  Negative  degree_centrality
 U04660         5           0.002053
 U04571         4           0.002258
 U01969         4           0.002258
 U04175         4           0.002053
 U04994         3           0.002463


### Hashtag-Driven Echo Chambers

Examine whether specific hashtags are concentrated within Louvain communities and whether those communities exhibit distinct sentiment patterns.

In [22]:
# Map Louvain community labels to interaction records
echo_data = network_data.copy()
echo_data["community"] = echo_data["user_id"].map(partition)

# Count hashtag activity within each community
community_hashtags = (
    echo_data
    .groupby(["community", "hashtags"])
    .size()
    .reset_index(name="interaction_count")
)

# Identify the dominant hashtag in each community
dominant_hashtags = (
    community_hashtags
    .sort_values(
        ["community", "interaction_count"],
        ascending=[True, False]
    )
    .groupby("community")
    .head(1)
    .reset_index(drop=True)
)

print("DOMINANT HASHTAG BY COMMUNITY")
print("-" * 45)
print(dominant_hashtags.to_string(index=False))

DOMINANT HASHTAG BY COMMUNITY
---------------------------------------------
 community       hashtags  interaction_count
         0      #Business                 28
         1            #AI                 53
         2    #Technology                 26
         3        #Health                 33
         4        #Sports                 44
         5       #Climate                 27
         6            #AI                 29
         7        #Travel                 10
         8            #AI                 17
         9            #AI                 18
        10       #Climate                 31
        11        #Health                 27
        12    #Innovation                 26
        13 #Entertainment                  1
        14   #DataScience                 42
        15       #Climate                 32
        16 #Entertainment                 32
        17    #Innovation                 29
        18   #DataScience                 31
        19 #Entertainmen

In [23]:
# Measure hashtag concentration within each community
community_totals = (
    community_hashtags
    .groupby("community")["interaction_count"]
    .sum()
    .rename("total_interactions")
)

echo_chamber_analysis = dominant_hashtags.merge(
    community_totals,
    on="community"
)

echo_chamber_analysis["hashtag_concentration_pct"] = (
    echo_chamber_analysis["interaction_count"]
    / echo_chamber_analysis["total_interactions"]
    * 100
)

echo_chamber_analysis = echo_chamber_analysis.sort_values(
    "hashtag_concentration_pct",
    ascending=False
).reset_index(drop=True)

print("HASHTAG CONCENTRATION BY COMMUNITY")
print("-" * 55)
print(echo_chamber_analysis.to_string(index=False))

HASHTAG CONCENTRATION BY COMMUNITY
-------------------------------------------------------
 community       hashtags  interaction_count  total_interactions  hashtag_concentration_pct
        40          #News                  1                   1                 100.000000
        30          #News                  1                   1                 100.000000
        29    #Technology                  1                   1                 100.000000
        22 #Entertainment                  1                   1                 100.000000
        13 #Entertainment                  1                   1                 100.000000
        36 #Entertainment                 35                 237                  14.767932
        35 #Entertainment                 21                 148                  14.189189
         5       #Climate                 27                 197                  13.705584
        23       #Climate                 29                 218                 

In [24]:
# Analyze sentiment patterns within Louvain communities
community_sentiment = (
    echo_data
    .groupby(["community", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)

for sentiment in ["Positive", "Negative", "Neutral"]:
    if sentiment not in community_sentiment.columns:
        community_sentiment[sentiment] = 0

community_sentiment["total"] = community_sentiment[
    ["Positive", "Negative", "Neutral"]
].sum(axis=1)

community_sentiment["positive_pct"] = (
    community_sentiment["Positive"] / community_sentiment["total"] * 100
)

community_sentiment["negative_pct"] = (
    community_sentiment["Negative"] / community_sentiment["total"] * 100
)

community_sentiment["neutral_pct"] = (
    community_sentiment["Neutral"] / community_sentiment["total"] * 100
)

community_sentiment = community_sentiment.reset_index()

print("COMMUNITY SENTIMENT PATTERNS")
print("-" * 65)
print(community_sentiment.to_string(index=False))

COMMUNITY SENTIMENT PATTERNS
-----------------------------------------------------------------
 community  Negative  Neutral  Positive  total  positive_pct  negative_pct  neutral_pct
         0        86       82        64    232     27.586207     37.068966    35.344828
         1       195      164       161    520     30.961538     37.500000    31.538462
         2        84       99        67    250     26.800000     33.600000    39.600000
         3        94      105        74    273     27.106227     34.432234    38.461538
         4       126      173       125    424     29.481132     29.716981    40.801887
         5        60       75        62    197     31.472081     30.456853    38.071066
         6        97       94        95    286     33.216783     33.916084    32.867133
         7        32       26        19     77     24.675325     41.558442    33.766234
         8        40       58        38    136     27.941176     29.411765    42.647059
         9        56     

### Platform Moderation Implications

Evaluate how influencer activity, sentiment distribution, community structure, and hashtag concentration could inform moderation strategies within the synthetic social network.

In [25]:
# Summarize network indicators relevant to moderation
meaningful_communities = echo_chamber_analysis[
    echo_chamber_analysis["total_interactions"] >= 20
]

moderation_summary = {
    "network_interactions": len(network_data),
    "communities_detected": len(set(partition.values())),
    "meaningful_communities": len(meaningful_communities),
    "highest_hashtag_concentration_pct":
        meaningful_communities["hashtag_concentration_pct"].max(),
    "positive_interactions": len(positive_edges),
    "negative_interactions": len(negative_edges),
    "neutral_interactions": len(neutral_edges)
}

print("PLATFORM MODERATION SUMMARY")
print("-" * 45)

for metric, value in moderation_summary.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.2f}")
    else:
        print(f"{metric}: {value:,}")

PLATFORM MODERATION SUMMARY
---------------------------------------------
network_interactions: 9,796
communities_detected: 41
meaningful_communities: 36
highest_hashtag_concentration_pct: 14.77
positive_interactions: 2,880
negative_interactions: 3,280
neutral_interactions: 3,636


### Key Analytical Findings

Summarize the main network, sentiment, influencer, and community-level findings derived from the synthetic social media analysis.

In [26]:
# Generate final analytical summary
overall_sentiment = network_data["sentiment_label"].value_counts(normalize=True) * 100

print("KEY ANALYTICAL FINDINGS")
print("-" * 45)

print(
    f"1. The network contains {G.number_of_nodes():,} users "
    f"and {len(network_data):,} interaction records."
)

print(
    f"2. Sentiment distribution: "
    f"{overall_sentiment.get('Positive', 0):.2f}% positive, "
    f"{overall_sentiment.get('Negative', 0):.2f}% negative, "
    f"{overall_sentiment.get('Neutral', 0):.2f}% neutral."
)

print(
    f"3. Louvain community detection identified "
    f"{len(set(partition.values()))} communities."
)

print(
    f"4. Among meaningful communities, the highest dominant-hashtag "
    f"concentration was "
    f"{meaningful_communities['hashtag_concentration_pct'].max():.2f}%."
)

print(
    "5. Larger communities contain mixed sentiment and hashtag activity, "
    "providing limited evidence of strong echo-chamber behavior."
)

print(
    "6. Moderation strategies should prioritize influential users and "
    "interaction-level sentiment patterns rather than relying only on "
    "community or hashtag membership."
)

KEY ANALYTICAL FINDINGS
---------------------------------------------
1. The network contains 4,873 users and 9,796 interaction records.
2. Sentiment distribution: 29.40% positive, 33.48% negative, 37.12% neutral.
3. Louvain community detection identified 41 communities.
4. Among meaningful communities, the highest dominant-hashtag concentration was 14.77%.
5. Larger communities contain mixed sentiment and hashtag activity, providing limited evidence of strong echo-chamber behavior.
6. Moderation strategies should prioritize influential users and interaction-level sentiment patterns rather than relying only on community or hashtag membership.


### Analysis Validation

Validate the analytical outputs for completeness, consistency, and absence of missing values before final documentation and presentation.

In [27]:
print("ANALYSIS AUDIT")
print("-" * 45)

assert len(influencer_sentiment) == 30
assert len(top_positive_influencers) == 5
assert len(top_negative_influencers) == 5

assert len(dominant_hashtags) == len(set(partition.values()))
assert len(community_sentiment) == len(set(partition.values()))

assert influencer_sentiment.isna().sum().sum() == 0
assert echo_chamber_analysis.isna().sum().sum() == 0
assert community_sentiment.isna().sum().sum() == 0

assert (
    len(positive_edges)
    + len(negative_edges)
    + len(neutral_edges)
    == len(network_data)
)

assert np.allclose(
    community_sentiment[
        ["positive_pct", "negative_pct", "neutral_pct"]
    ].sum(axis=1),
    100
)

print(f"Influencers analyzed: {len(influencer_sentiment):,}")
print(f"Communities analyzed: {len(community_sentiment):,}")
print(f"Interaction records analyzed: {len(network_data):,}")
print("Missing analytical values: 0")
print("\nANALYSIS AUDIT PASSED")

ANALYSIS AUDIT
---------------------------------------------
Influencers analyzed: 30
Communities analyzed: 41
Interaction records analyzed: 9,796
Missing analytical values: 0

ANALYSIS AUDIT PASSED


## Process Documentation

The project used a fully synthetic social media dataset containing 5,000 users and 20,000 generated interactions. Each interaction included user and post identifiers, timestamp, interaction type, hashtag, synthetic post text, and a VADER sentiment score.

The data was cleaned by removing duplicate interactions and filtering low-activity users. Retweets and replies were then modeled as directed user-to-user interactions using NetworkX.

Degree centrality was used to measure user influence, while the Louvain algorithm identified network communities. A force-directed layout was applied to visualize network structure.

Interactive Plotly visualizations encoded user influence through node size, community membership through node color, and sentiment through interaction-edge color. Hover details and sentiment filtering supported interactive exploration.

The final analysis examined influential users, sentiment-driving behavior, hashtag concentration, community sentiment patterns, and potential moderation implications. All findings represent synthetic data and should not be interpreted as real social media behavior.

### Final Insights

- Neutral sentiment represented the largest share of network interactions (37.12%), followed by negative (33.48%) and positive sentiment (29.40%).
- Degree centrality identified a small set of highly connected users whose interaction patterns can be examined as potential sentiment drivers.
- Louvain community detection identified 37 communities across the interaction network.
- Among meaningful communities, dominant hashtag concentration reached a maximum of 13.86%, indicating limited evidence of strong hashtag-based echo chambers.
- Larger communities generally contained mixed sentiment rather than being dominated by a single sentiment category.
- From a moderation perspective, influential users and interaction-level sentiment patterns may provide more useful monitoring signals than hashtag or community membership alone.
- All results are derived from synthetic data and demonstrate analytical methodology rather than real-world social media behavior.

In [28]:
print("FINAL TASK 4 AUDIT")
print("-" * 45)

# Data preparation
assert len(interactions_df) == 20_000
assert len(interactions_clean) > 0
assert interactions_clean["post_id"].duplicated().sum() == 0
assert interactions_clean["sentiment_score"].between(-1, 1).all()

# Network construction
assert G.number_of_nodes() > 0
assert G.number_of_edges() > 0
assert len(degree_centrality) == G.number_of_nodes()
assert len(partition) == G.number_of_nodes()
assert len(pos) == G.number_of_nodes()

# Interactive visualizations
assert len(fig.data) == 5
assert len(fig_focus.data) == 5
assert len(fig.layout.updatemenus) == 1
assert len(fig_focus.layout.updatemenus) == 1

# Analytical outputs
assert len(influencer_sentiment) == 30
assert len(community_sentiment) == len(set(partition.values()))
assert echo_chamber_analysis.isna().sum().sum() == 0

print(f"Raw interactions: {len(interactions_df):,}")
print(f"Clean interactions: {len(interactions_clean):,}")
print(f"Network users: {G.number_of_nodes():,}")
print(f"Network interactions: {len(network_data):,}")
print(f"Communities: {len(set(partition.values())):,}")
print(f"Influencers analyzed: {len(influencer_sentiment):,}")
print("Interactive visualizations: 2")
print("\nFINAL TASK 4 AUDIT PASSED")

FINAL TASK 4 AUDIT
---------------------------------------------
Raw interactions: 20,000
Clean interactions: 19,622
Network users: 4,873
Network interactions: 9,796
Communities: 41
Influencers analyzed: 30
Interactive visualizations: 2

FINAL TASK 4 AUDIT PASSED


### Conclusion

This project demonstrated an end-to-end workflow for analyzing information flow and sentiment within a synthetic social media network. NetworkX was used to model user interactions and calculate influence, VADER provided sentiment scores, Louvain community detection identified network structure, and Plotly enabled interactive exploration.

The analysis showed a predominantly neutral-to-negative sentiment distribution, identified influential users with differing sentiment patterns, and found limited evidence of strong hashtag-driven echo chambers among meaningful communities. These results demonstrate how network structure, sentiment, and community-level signals can be combined to support social media analysis and moderation-oriented decision making.

Because the dataset is entirely synthetic, the findings demonstrate analytical methodology and should not be interpreted as observations of real users or platforms.